# 02 - Model Training (Multi-Config)
## Loop 4 configs: 10K, 30K, 50K, 100K

Input: `cleaned_train_*K.pkl` dari notebook 01
Output: models + feature importance + CV results per config

In [ ]:
import pandas as pd
import numpy as np
import pickle, time, os, gc
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

CONFIGS = [10000, 30000, 50000, 100000]
DATA_DIR = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

all_results = {}
print('OK')

In [ ]:
for config in CONFIGS:
    suffix = f'{config//1000}K'
    pkl_path = os.path.join(DATA_DIR, f'cleaned_train_{suffix}.pkl')
    
    if not os.path.exists(pkl_path):
        print(f'SKIP {suffix}: file not found')
        continue
    
    print(f'\n{"="*70}')
    print(f'  CONFIG: {suffix}')
    print(f'{"="*70}')
    
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    X_train = data['X']
    y_raw = data['y']
    feature_names = data['feature_names']
    all_classes = data['classes']
    
    # Re-encode
    unique_labels = sorted(np.unique(y_raw))
    classes = [all_classes[i] for i in unique_labels]
    label_map = {old: new for new, old in enumerate(unique_labels)}
    y_train = np.array([label_map[y] for y in y_raw])
    NUM_CLASSES = len(classes)
    print(f'Samples: {len(X_train):,} | Features: {X_train.shape[1]} | Classes: {NUM_CLASSES}')
    
    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = ['accuracy', 'f1_macro']
    
    xgb_model = XGBClassifier(objective='multi:softmax', num_class=NUM_CLASSES,
        max_depth=6, learning_rate=0.3, n_estimators=100,
        eval_metric='mlogloss', random_state=42, n_jobs=-1)
    
    print('  XGBoost CV...', end=' ')
    t0 = time.time()
    xgb_cv = cross_validate(xgb_model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    print(f'{time.time()-t0:.1f}s')
    
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    print('  RF CV...', end=' ')
    t0 = time.time()
    rf_cv = cross_validate(rf_model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    print(f'{time.time()-t0:.1f}s')
    
    xgb_acc = xgb_cv['test_accuracy'].mean()*100
    xgb_f1 = xgb_cv['test_f1_macro'].mean()*100
    rf_acc = rf_cv['test_accuracy'].mean()*100
    rf_f1 = rf_cv['test_f1_macro'].mean()*100
    print(f'  XGBoost: Acc={xgb_acc:.2f}% F1={xgb_f1:.2f}%')
    print(f'  RF:      Acc={rf_acc:.2f}% F1={rf_f1:.2f}%')
    
    # Train final + Top-10 model
    print('  Training final model...')
    xgb_final = XGBClassifier(objective='multi:softmax', num_class=NUM_CLASSES,
        max_depth=6, learning_rate=0.3, n_estimators=100,
        eval_metric='mlogloss', random_state=42, n_jobs=-1)
    xgb_final.fit(X_train, y_train)
    
    # Feature importance
    importance = xgb_final.feature_importances_
    feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
    feat_imp = feat_imp.sort_values('Importance', ascending=False).reset_index(drop=True)
    top_10 = feat_imp.head(10)['Feature'].tolist()
    print(f'  Top-10: {top_10}')
    
    # Save full model
    full_path = os.path.join(MODEL_DIR, f'xgboost_full_{suffix}.json')
    xgb_final.save_model(full_path)
    full_size = os.path.getsize(full_path) / 1024
    
    # Train Top-10 model
    X_top10 = X_train[top_10]
    xgb_top10 = XGBClassifier(objective='multi:softmax', num_class=NUM_CLASSES,
        max_depth=6, learning_rate=0.3, n_estimators=100,
        eval_metric='mlogloss', random_state=42, n_jobs=-1)
    xgb_top10.fit(X_top10, y_train)
    top10_path = os.path.join(MODEL_DIR, f'xgboost_top10_{suffix}.json')
    xgb_top10.save_model(top10_path)
    top10_size = os.path.getsize(top10_path) / 1024
    
    # Inference time
    booster = xgb_top10.get_booster()
    dm = xgb.DMatrix(X_top10.iloc[:100])
    times = [(time.time(), booster.predict(dm))[0] for _ in range(10)]
    # proper timing
    inf_times = []
    for _ in range(10):
        t0 = time.time()
        booster.predict(dm)
        inf_times.append((time.time()-t0)*1000)
    avg_inf = np.mean(inf_times)
    
    print(f'  Full model: {full_size:.1f} KB | Top-10: {top10_size:.1f} KB | Inf: {avg_inf:.1f}ms')
    
    # Save results
    all_results[suffix] = {
        'xgb_acc': xgb_acc, 'xgb_f1': xgb_f1,
        'rf_acc': rf_acc, 'rf_f1': rf_f1,
        'top_10': top_10, 'classes': classes,
        'label_map': label_map, 'num_classes': NUM_CLASSES,
        'full_model_kb': full_size, 'top10_model_kb': top10_size,
        'inference_ms': avg_inf, 'samples': len(X_train)
    }
    feat_imp.to_csv(os.path.join(DATA_DIR, f'feature_importance_{suffix}.csv'), index=False)
    
    del X_train, y_train, data; gc.collect()

# Save all results
with open(os.path.join(DATA_DIR, 'training_results_all.pkl'), 'wb') as f:
    pickle.dump(all_results, f)
print(f'\n{"="*70}')
print('ALL DONE. Proceed to notebook 03.')

In [ ]:
# Summary table
print(f'{"Config":>8s} {"Samples":>10s} {"XGB Acc%":>10s} {"XGB F1%":>10s} {"RF Acc%":>10s} {"RF F1%":>10s} {"Model KB":>10s}')
print('-'*70)
for k, v in all_results.items():
    print(f'{k:>8s} {v["samples"]:>10,} {v["xgb_acc"]:>10.2f} {v["xgb_f1"]:>10.2f} {v["rf_acc"]:>10.2f} {v["rf_f1"]:>10.2f} {v["top10_model_kb"]:>10.1f}')